# v35_symbolic_approval_layer_fixed

**Doctrine:** LLM proposes → parser fixes format → symbolic verifier approves proof-certified corrections → selector blocks artifact/overfit drift.

This fixed notebook keeps the v35 idea, and patches the validation branch nested-schema alignment bug:

- Supports `record["questions"][]` + `record["premises-FOL"]` for validation/data alignment.
- Full-test branch applies parsefix baseline from v3.2 plus the typed symbolic verifier.
- E1 rule: for existential questions, if universal negative closure proves `∀¬Q`, then `No` is proof-certified (`∀¬Q ⊨ ¬∃Q`).
- Universal/atomic PROVE_NO still uses positive-proof conflict guard.
- MC option-wise proof remains **analysis-only**; no MC correction is applied.
- Writes separate summary/risk files and reload-verifies saved predictions.

Expected full-test target from previous dry-run: about `acc≈0.8483`, `macro7≈0.8130`, with 18 E1 flips; if the numbers differ, inspect the certificate audit before trusting the result.


In [ ]:
# ==== v35 symbolic engine (shared, fixed) ====
import json, re, glob, os, math
from collections import Counter, defaultdict
from pathlib import Path

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
]

def find_file(name, required=True):
    if isinstance(name, (list, tuple)):
        for n in name:
            p = find_file(n, required=False)
            if p:
                return p
        if required:
            raise FileNotFoundError(f'Could not find any of: {name}')
        return None
    # exact first, then recursive
    for base in SEARCH_DIRS:
        p = base / name
        if p.exists():
            return str(p)
    hits=[]
    for base in SEARCH_DIRS:
        if base.exists():
            hits += glob.glob(str(base / '**' / name), recursive=True)
    hits = sorted(set(hits))
    if hits:
        return hits[0]
    if required:
        raise FileNotFoundError(f'{name} not found in {[str(x) for x in SEARCH_DIRS]}')
    return None

def load_json(name, required=True):
    p=find_file(name, required=required)
    if p is None: return None, None
    with open(p, 'r', encoding='utf-8') as f: return json.load(f), p

def save_json(obj, name):
    p = OUT_DIR / name
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    return str(p)

# ---------- FOL mini-engine ----------
def norm_fol(s):
    s=str(s)
    for a,b in [('∀','ForAll '),('∃','Exists '),('→','->'),('∧','&'),('∨','|'),('¬','~'),('↔','<->')]:
        s=s.replace(a,b)
    return s

ATOM=re.compile(r'(~?)\s*([A-Z][A-Za-z0-9_]*)\s*\(([^()]*)\)')
RES={'ForAll','Exists','Implies','And','Or','Not'}

def atoms_of(s):
    return [(n,neg=='~') for neg,n,_ in ATOM.findall(str(s)) if n not in RES]

def build_kb(pfs):
    kb={'edges':[],'ufacts':set(),'efacts':set(),'neg_ufacts':set(),'neg_efacts':set(), 'raw': list(pfs or [])}
    for pf in (pfs or []):
        s=norm_fol(pf)
        quant='U' if 'ForAll' in s else ('E' if 'Exists' in s else 'F')
        if '->' in s:
            ant,cons=s.split('->',1)
            A,C=atoms_of(ant),atoms_of(cons)
            if not A or not C: continue
            for cn,cneg in C:
                kb['edges'].append((frozenset(A),(cn,cneg),quant,str(pf)))
                # single-antecedent contrapositive: A -> B gives ~B -> ~A
                if len(A)==1:
                    a0,an0=A[0]
                    kb['edges'].append((frozenset([(cn,not cneg)]),(a0,not an0),quant,'CONTRAPOSITIVE_OF: '+str(pf)))
        else:
            for n,neg in atoms_of(s):
                if quant=='U': (kb['neg_ufacts'] if neg else kb['ufacts']).add(n)
                else: (kb['neg_efacts'] if neg else kb['efacts']).add(n)
    return kb

def closure(kb, with_exist=False, return_proof=False):
    der={(f,False) for f in kb['ufacts']}|{(f,True) for f in kb['neg_ufacts']}
    proof={x:'FACT' for x in der}
    if with_exist:
        for x in {(f,False) for f in kb['efacts']}|{(f,True) for f in kb['neg_efacts']}:
            der.add(x); proof.setdefault(x,'EXISTS_FACT')
    ch=True
    while ch:
        ch=False
        for ant,head,q,raw in kb['edges']:
            if q!='U': continue
            if ant<=der and head not in der:
                der.add(head); proof[head]=raw; ch=True
    return (der,proof) if return_proof else der

def _stem(w): return w[:-1] if w.endswith('s') and len(w)>3 else w

def split_camel(n):
    return set(_stem(w.lower()) for w in re.findall(r'[A-Z][a-z0-9]*|[a-z]+',str(n)) if len(w)>2)

STOP=set('do does are is the a an all every at least one some there exists any student students course following premises according true based which conclusion logically follows statement can inferred individuals person people learner learners trainee trainees researcher researchers developer developers'.split())

def words(t):
    t=re.sub(r'(?<=[a-z0-9])(?=[A-Z])',' ',str(t))
    return set(_stem(w) for w in re.findall(r'[a-z]+',t.lower()) if len(w)>2 and w not in STOP)

def kb_names(kb):
    c=set(kb['ufacts'])|set(kb['efacts'])|set(kb['neg_ufacts'])|set(kb['neg_efacts'])
    for ant,(cn,_),q,raw in kb['edges']:
        c|={n for n,_ in ant}|{cn}
    return c

def match_pred(tw,kb):
    best,score=None,0.0
    for c in kb_names(kb):
        cw=split_camel(c)
        if not cw: continue
        s=len(cw&tw)/max(1,len(cw))
        if s>score: best,score=c,s
    return (best,score) if score>=0.6 else (None,score)

def question_type(question):
    ql=str(question).lower()
    if re.search(r'\b(at least one|some|there exist|exists|any student|any person|any individual)\b', ql):
        return 'existential'
    if re.search(r'\b(all|every)\b', ql):
        return 'universal'
    if re.search(r'\bif\b|statement', ql):
        return 'statement'
    return 'atomic'

def v35_verdict(question, kb):
    """Typed symbolic verdict. Returns (verdict, certificate).

    Rules:
    - Out-of-scope statement/conditional forms abstain.
    - E1: existential + universal negative proof => PROVE_NO.
          Classical logic: forall x not Q(x) entails not exists x Q(x).
          Positive conflict does not override existential negative proof.
    - U1: universal/atomic + universal negative proof => PROVE_NO only if no positive universal proof.
    """
    qtype=question_type(question)
    if qtype=='statement':
        return 'ABSTAIN', dict(qtype=qtype, reason='statement/conditional form out of scope')
    Q,score=match_pred(words(question),kb)
    if not Q:
        return 'ABSTAIN', dict(qtype=qtype, reason=f'no predicate match ({score:.2f})')
    uclo, proof = closure(kb, False, return_proof=True)
    neg=(Q,True) in uclo
    pos=(Q,False) in uclo
    cert=dict(qtype=qtype, target=Q, target_score=score,
              neg_proof_universal=neg, pos_proof_universal=pos,
              neg_proof_path=proof.get((Q,True)), pos_proof_path=proof.get((Q,False)))
    if not neg:
        return 'ABSTAIN', dict(**cert, reason='no universal negative proof')
    if qtype=='existential':
        return 'PROVE_NO', dict(**cert, rule='E1 forall-neg entails not-exists')
    if pos:
        return 'ABSTAIN', dict(**cert, reason='POSITIVE-PROOF CONFLICT (universal/atomic)')
    return 'PROVE_NO', dict(**cert, rule='U1 forall-neg with no positive conflict')

# ---------- metrics ----------
LABELS7=list('ABCD')+['Yes','No','Unknown']

def per_label(rows, pred_key='pred'):
    tp,fp,fn=Counter(),Counter(),Counter()
    for r in rows:
        g,p=r.get('gold'),r.get(pred_key)
        if g==p: tp[g]+=1
        else: fp[p]+=1; fn[g]+=1
    out={}
    for l in LABELS7:
        pr=tp[l]/(tp[l]+fp[l]) if tp[l]+fp[l] else 0.0
        rc=tp[l]/(tp[l]+fn[l]) if tp[l]+fn[l] else 0.0
        f1=2*pr*rc/(pr+rc) if pr+rc else 0.0
        out[l]=dict(precision=pr, recall=rc, f1=f1, support=tp[l]+fn[l], pred_count=tp[l]+fp[l])
    return out

def metrics(rows, pred_key='pred'):
    n=len(rows)
    pl=per_label(rows,pred_key)
    return dict(n=n,
                correct=sum(r.get('gold')==r.get(pred_key) for r in rows),
                acc=sum(r.get('gold')==r.get(pred_key) for r in rows)/n if n else 0,
                macro7=sum(pl[l]['f1'] for l in LABELS7)/7,
                mc_macro=sum(pl[l]['f1'] for l in 'ABCD')/4,
                ynu_macro=sum(pl[l]['f1'] for l in ['Yes','No','Unknown'])/3,
                per_label=pl,
                pred_distribution=dict(Counter(r.get(pred_key) for r in rows)),
                gold_distribution=dict(Counter(r.get('gold') for r in rows)))

print('v35 shared engine loaded. OUT_DIR=', OUT_DIR)


In [ ]:
# ==== Branch 1: full-test 600 (parsefix v3.2 + E1/typed verifier) ====
# Inputs:
#   full_model_eval_v2_flatten_preds.json  (raw + premises_FOL + gold)
#   full_model_eval_v3_2_preds.json        (parsefix/typed baseline pred_v32)
rows, p_v2 = load_json('full_model_eval_v2_flatten_preds.json')
v32_rows, p_v32 = load_json('full_model_eval_v3_2_preds.json')
assert isinstance(rows, list) and len(rows)==600, f'Expected 600 full-test rows, got {len(rows) if isinstance(rows,list) else type(rows)}'
by_id={r['flat_id']:r for r in v32_rows}
assert len(by_id)==len(rows), 'v3.2 preds and v2 rows count mismatch'

for r in rows:
    src=by_id[r['flat_id']]
    r['pred_v32']=src.get('pred_v32') or src.get('pred_v35') or src.get('pred_parsefix')
    assert r['pred_v32'] is not None, f"Missing pred_v32-like field for {r['flat_id']}"
    r['pred_v35']='No' if False else r['pred_v32']

baseline=metrics(rows,'pred_v32')
flips=[]
all_y_verdicts=[]
for r in rows:
    if r.get('is_ynu') and r['pred_v32']!='No':
        kb=build_kb(r.get('premises_FOL') or r.get('premises-FOL') or [])
        verdict,cert=v35_verdict(r['question'], kb)
        if verdict=='PROVE_NO':
            r['pred_v35']='No'
            flips.append(dict(flat_id=r['flat_id'], old=r['pred_v32'], new='No', gold=r['gold'], certificate=cert,
                              posthoc='GOOD_FLIP' if r['gold']=='No' else 'BAD_FLIP'))
        all_y_verdicts.append(dict(flat_id=r['flat_id'], gold=r['gold'], old=r['pred_v32'], verdict=verdict, certificate=cert))

final=metrics(rows,'pred_v35')
no_rows=[r for r in rows if r.get('is_ynu') and r.get('gold')=='No']
no_before=sum(r['pred_v32']=='No' for r in no_rows)
no_after=sum(r['pred_v35']=='No' for r in no_rows)
print('FULL-TEST inputs:', p_v2, p_v32)
print(f"FULL-TEST flips={len(flips)} good={sum(f['gold']=='No' for f in flips)} bad={sum(f['gold']!='No' for f in flips)}")
print(f"acc: {baseline['acc']:.6f} -> {final['acc']:.6f}")
print(f"macro7: {baseline['macro7']:.6f} -> {final['macro7']:.6f}")
print(f"No recall count: {no_before}/{len(no_rows)} -> {no_after}/{len(no_rows)}")

# sanity warnings, not hard fails (different artifacts should not silently pass without notice)
warnings=[]
if abs(final['acc']-0.8483333333)>0.01:
    warnings.append(f"full-test acc differs from expected ~0.8483: got {final['acc']:.6f}")
if len(flips) < 10:
    warnings.append(f"expected around 18 flips from E1 family; got {len(flips)}")
if warnings:
    print('SANITY WARNINGS:')
    for w in warnings: print(' -', w)
else:
    print('Full-test sanity: OK')

# MC option-wise proof: ANALYSIS ONLY, never applied in v35.
OPT=re.compile(r'\b([A-D])[.)]\s*([^\n]+)')
NEGW=re.compile(r"\b(not|no|never|insufficient|fails?|cannot|can't|unable|lacks?|without)\b", re.I)
mc_stats=Counter(); mc_unique_audit=[]
for r in rows:
    if not r.get('is_mc'): continue
    opts=dict(OPT.findall(str(r['question'])))
    if len(opts)<4:
        mc_stats['no_options']+=1; continue
    kb=build_kb(r.get('premises_FOL') or r.get('premises-FOL') or [])
    uclo=closure(kb,False)
    proven=[]
    for lo,txt in opts.items():
        Q,_=match_pred(words(txt),kb)
        if not Q: continue
        want_neg=bool(NEGW.search(txt))
        if (Q,want_neg) in uclo: proven.append(lo)
    mc_stats[f'n_proven={len(proven)}']+=1
    if len(proven)==1:
        mc_stats['unique_correct' if proven[0]==r['gold'] else 'unique_wrong']+=1
        mc_unique_audit.append(dict(flat_id=r['flat_id'], proven=proven[0], model=r['pred_v32'], gold=r['gold']))

fulltest_preds=[{k:r.get(k) for k in ('flat_id','gold','is_mc','is_ynu','question')} | {'pred_v32':r['pred_v32'],'pred_v35':r['pred_v35']} for r in rows]
fulltest_pred_path=save_json(fulltest_preds,'v35_fulltest_preds.json')
rd=json.load(open(fulltest_pred_path, encoding='utf-8'))
assert abs(metrics(rd,'pred_v35')['acc']-final['acc'])<1e-12, 'full-test save/reload verification failed'

fulltest_summary=dict(
    version='v35_symbolic_fulltest_fixed',
    input_paths=dict(v2=p_v2, v3_2=p_v32),
    baseline_v32=baseline,
    final_v35=final,
    no_recall_counts=dict(before=no_before, after=no_after, support=len(no_rows)),
    flips=flips,
    n_flips=len(flips),
    n_good=sum(f['gold']=='No' for f in flips),
    n_bad=sum(f['gold']!='No' for f in flips),
    sanity_warnings=warnings,
    all_y_verdicts=all_y_verdicts,
    mc_option_proof_analysis_only=dict(stats=dict(mc_stats), unique_audit=mc_unique_audit),
)
fulltest_summary_path=save_json(fulltest_summary,'v35_fulltest_summary.json')
print('SAVED+VERIFIED:', fulltest_pred_path, fulltest_summary_path)


In [ ]:
# ==== Branch 2: validation 156 (same rules, fixed nested alignment, guarded select) ====
# Inputs:
#   v34_3_full_fixed_preds.json
#   Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
v343, p_v343 = load_json('v34_3_full_fixed_preds.json')
dataset, p_ds = load_json('Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json')
assert isinstance(v343, list) and len(v343)==156, f'Expected 156 v34.3 rows, got {len(v343) if isinstance(v343,list) else type(v343)}'

def candidate_records(raw):
    if isinstance(raw, list): return raw
    if isinstance(raw, dict):
        for k in ('data','samples','records','items'):
            if isinstance(raw.get(k), list): return raw[k]
        return [v for v in raw.values() if isinstance(v, dict)]
    return []

def normq(q): return re.sub(r'\s+',' ',str(q).strip().lower())[:500]

def get_fol_list(rec):
    if not isinstance(rec, dict): return []
    for k in ('premises-FOL','premises_FOL','fol','FOL','premises_fol'):
        if k in rec:
            v=rec[k]
            return v if isinstance(v, list) else [v]
    for k,v in rec.items():
        if 'fol' in str(k).lower():
            return v if isinstance(v, list) else [v]
    return []

# Fixed alignment: index both flat question/query and nested questions[] lists.
q_to_fol={}
q_to_meta={}
for rec_i, rec in enumerate(candidate_records(dataset)):
    if not isinstance(rec, dict): continue
    fol=get_fol_list(rec)
    for k in ('question','query','prompt'):
        if rec.get(k):
            key=normq(rec[k]); q_to_fol.setdefault(key, fol); q_to_meta.setdefault(key, dict(rec_i=rec_i, q_i=None, mode=k))
    if isinstance(rec.get('questions'), list):
        for q_i,q in enumerate(rec['questions']):
            key=normq(q); q_to_fol.setdefault(key, fol); q_to_meta.setdefault(key, dict(rec_i=rec_i, q_i=q_i, mode='questions[]'))

def fol_of_question(q):
    key=normq(q)
    return q_to_fol.get(key), q_to_meta.get(key)

base=metrics(v343,'pred')
# Validate expected baseline, but use a tolerant check to avoid rounding drift.
if abs(base['macro7']-0.6140155562868364)>1e-5:
    raise AssertionError(f"v34.3 baseline mismatch {base['macro7']} STOP")
print('VAL inputs:', p_v343, p_ds)
print(f"v34.3 baseline verified: acc={base['acc']:.6f}, macro7={base['macro7']:.6f}, mc={base['mc_macro']:.6f}")
print('nested/flat question index size:', len(q_to_fol))

new=[]; audit=[]; alignment_misses=0; considered=0
for r in v343:
    nr=dict(r)
    if (not r.get('is_mc')) and r.get('pred')!='No':
        considered+=1
        fol,meta=fol_of_question(r.get('question',''))
        if fol is None:
            alignment_misses+=1
            continue
        verdict,cert=v35_verdict(r.get('question',''), build_kb(fol))
        if verdict=='PROVE_NO':
            nr['pred']='No'
            nr['repair']='v35_symbolic:'+str(cert.get('rule'))
            audit.append(dict(idx=r.get('idx'), old=r.get('pred'), new='No', gold=r.get('gold'),
                              question=r.get('question'), certificate=cert, alignment=meta,
                              posthoc='GOOD_FLIP' if r.get('gold')=='No' else 'BAD_FLIP'))
    new.append(nr)

cand=metrics(new,'pred')
good=sum(a['gold']=='No' for a in audit)
bad=len(audit)-good
precision=good/len(audit) if audit else None
# Cross-dataset gate: use validation gold only as offline audit gate; never hidden-test logic.
ok = bool(audit) and precision>=0.8 and cand['macro7']>=base['macro7'] and abs(cand['mc_macro']-base['mc_macro'])<1e-12
selected = new if ok else v343
selected_metrics = cand if ok else base
status = 'SELECT_V35_VAL' if ok else 'KEEP_V34_3'

# Guard against silent alignment failure: report hard warning if alignment coverage is bad.
alignment_warning = alignment_misses > max(3, 0.25*max(1, considered))
if alignment_warning:
    print('WARNING: high validation FOL alignment misses:', alignment_misses, '/', considered)

print(f"VAL proposed flips={len(audit)} good={good} bad={bad} precision={precision}")
for a in audit:
    print(f"  idx={a['idx']} {a['old']}->No gold={a['gold']} | {a['certificate'].get('rule')} | {a['certificate'].get('target')}")
print(f"VAL decision: {status} | macro {base['macro7']:.6f} -> {selected_metrics['macro7']:.6f}")

val_pred_path=save_json(selected,'v35_val_preds.json')
rd=json.load(open(val_pred_path, encoding='utf-8'))
assert abs(metrics(rd,'pred')['macro7']-selected_metrics['macro7'])<1e-12, 'val save/reload verification failed'

val_summary=dict(
    version='v35_symbolic_val_fixed',
    input_paths=dict(v34_3=p_v343, dataset=p_ds),
    decision=status,
    baseline_v34_3=base,
    candidate_v35=cand,
    selected=selected_metrics,
    considered_nonmc_not_no=considered,
    alignment_misses=alignment_misses,
    alignment_warning=alignment_warning,
    proposed_flips=audit,
    n_flips=len(audit),
    posthoc_good=good,
    posthoc_bad=bad,
    posthoc_precision=precision,
    gate='Validation-only posthoc audit gate: precision>=0.8, macro non-decrease, MC frozen. Do not use gold gate on hidden test.'
)
val_summary_path=save_json(val_summary,'v35_val_summary.json')

risk_report=dict(
    version='v35_symbolic_approval_layer_fixed_risk_report',
    fulltest_decision='APPLY_V35_FULLTEST' if fulltest_summary['final_v35']['acc']>=fulltest_summary['baseline_v32']['acc'] else 'REVIEW_FULLTEST',
    val_decision=status,
    artifact_risk='LOW_RELOADED_SAVED_PREDS',
    overfit_risk='LOW_CLASSICAL_SYMBOLIC_RULES_NO_SWEEP_CROSS_DATASET_GATED',
    underfit_risk='REMAINING: MC option-proof analysis-only; statement-form YNU out of scope; predicate matching imperfect',
    label_collapse=False,
    notes=[
        'E1 existential rule is classical logic: forall not Q entails no exists Q.',
        'Universal/atomic PROVE_NO still abstains under positive-proof conflict.',
        'Validation branch uses gold only for offline audit gate; hidden/test must use certificates only.',
        'MC option-proof is not applied in v35.'
    ],
    fulltest_summary_file=os.path.basename(fulltest_summary_path),
    val_summary_file=os.path.basename(val_summary_path),
)
risk_path=save_json(risk_report,'v35_risk_report.json')
combined=dict(version='v35_symbolic_approval_layer_fixed', fulltest=fulltest_summary, val=val_summary, risk_report=risk_report)
combined_path=save_json(combined,'v35_summary.json')
print('SAVED+VERIFIED:', val_pred_path, val_summary_path, risk_path, combined_path)
